In [1]:
import pandas as pd
import numpy as np
from scipy import stats

from pandas.api.types import CategoricalDtype

stationProximityDataSet = pd.read_csv("data/benz-tr.csv")

# Drop rows where country is not czechia

stationProximityDataSet = stationProximityDataSet[stationProximityDataSet.country == "cze"]

major_cities = ["praha", "brno", "ostrava", "liberec", "plzen", 
                "olomouc", "budejovice", "hradec kralove", 
                "pardubice", "usti nad labem", "zlin", "havirov", "kladno",
                "most", "opava", "jihlava", "frydek-mistek", "teplice", "karvina", "karlovy vary",
                "chomutov", "decin", "mlada boleslav", "jablonec nad nisou", "prostejov", "prerov", "trinec" ]

# without cities

stationProximityDataSet = stationProximityDataSet[~stationProximityDataSet['city'].str.lower().isin(major_cities)]


print("\nCount of transactions in Czechia: ", stationProximityDataSet.shape[0])

# Lets consider rows which have the same amnt, day, hour, clid to be duplicate and drop them
duplicateSubset = subset=["amt", "day", "hour", "clid"]
print('\nCount of duplicated rows: ',
      sum(stationProximityDataSet.duplicated(subset=duplicateSubset)))

stationProximityDataSet = stationProximityDataSet.drop_duplicates(subset=duplicateSubset)

# Drop useless columns
stationProximityDataSet = stationProximityDataSet.drop(columns=["clst", "clid", "day", "hour", "fav", "fav_main", "cl_gps_lat", "cl_gps_lon", "city", "name", "dist", "country"])

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

duplicateSubset = ["gps_lat", "gps_lon"]

print('\nCount of duplicated gps locations: ',
      sum(stationsWithDifferingGPS.duplicated(subset=duplicateSubset)))
# This is problematic because some stations share same gps location
# But considering that the gps is in most cases if not all cases precise for 5-6 decimal points which is 0.1m precision, we will consider them as one station

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

stationsWithDifferingGPS = stationsWithDifferingGPS.groupby(['gps_lat', 'gps_lon'], as_index=False).agg(
    total_amt=('total_amt', 'sum'),
    posid=('posid', 'first')
)

print('Count of stations', stationsWithDifferingGPS.shape[0])
stationsWithDifferingGPS


Count of transactions in Czechia:  1313151

Count of duplicated rows:  572

Count of duplicated gps locations:  597
Count of stations 1920


,gps_lat,gps_lon,total_amt,posid
0,48.587223,14.322222,17905.70,8062
1,48.619080,14.324732,106154.00,8685
2,48.624376,14.248910,13322.30,6390
3,48.637203,14.230956,134472.40,6440
4,48.649460,14.452580,2319653.78,5090
...,...,...,...,...
1915,51.000679,14.575062,4610.02,5712
1916,51.002682,14.464286,187055.20,7936
1917,51.004217,14.456016,1946.80,7939
1918,51.024219,14.456202,5294.10,7937


In [2]:
from geopy.distance import geodesic

major_cities_coords = {
    'praha': (50.073658, 14.418540),
    'brno': (49.195061, 16.606836),
    'ostrava': (49.820923, 18.262524),
    'liberec': (50.766280, 15.054339),
    'plzen': (49.738430, 13.373637),
    'olomouc': (49.593777, 17.250879),
    'budejovice': (48.975658, 14.480255),
    'hradec kralove': (50.210361, 15.825211),
    'pardubice': (50.034309, 15.781199),
    'usti nad labem': (50.661116, 14.053146),
    'zlin': (49.224437, 17.662763),
    'havirov': (49.780392, 18.430625),
    'kladno': (50.141699, 14.106746),
    'most': (50.503010, 13.636171),
    'opava': (49.938662, 17.902086),
    'jihlava': (49.396025, 15.591246),
    'frydek-mistek': (49.685349, 18.350273),
    'teplice': (50.640103, 13.824016),
    'karvina': (49.854042, 18.541672),
    'karlovy vary': (50.231689, 12.871006),
    'chomutov': (50.460037, 13.417780),
    'decin': (50.782186, 14.214780),
    'mlada boleslav': (50.411376, 14.903328),
    'jablonec nad nisou': (50.722997, 15.170052),
    'prostejov': (49.472423, 17.111664),
    'prerov': (49.455212, 17.450103),
    'trinec': (49.677116, 18.670659),
}


exclusion_radius_km = 30

def is_near_major_city(lat, lon, cities_coords, radius_km):
    for city, coords in cities_coords.items():
        if geodesic((lat, lon), coords).km <= radius_km:
            return True
    return False

stationsWithDifferingGPS['near_major_city'] = stationsWithDifferingGPS.apply(
    lambda row: is_near_major_city(row['gps_lat'], row['gps_lon'], major_cities_coords, exclusion_radius_km),
    axis=1
)

stationsWithDifferingGPS = stationsWithDifferingGPS[~stationsWithDifferingGPS['near_major_city']]

stationsWithDifferingGPS = stationsWithDifferingGPS.drop(columns='near_major_city')

In [3]:
stationsWithDifferingGPS

,gps_lat,gps_lon,total_amt,posid
0,48.587223,14.322222,17905.70,8062
1,48.619080,14.324732,106154.00,8685
2,48.624376,14.248910,13322.30,6390
3,48.637203,14.230956,134472.40,6440
4,48.649460,14.452580,2319653.78,5090
...,...,...,...,...
1910,50.984135,14.597880,295.40,5713
1915,51.000679,14.575062,4610.02,5712
1916,51.002682,14.464286,187055.20,7936
1918,51.024219,14.456202,5294.10,7937


In [4]:
#For station pairs I will try to do network graph. lets start by excluding useless data for this graph
from geopy import distance

# calculate geo distance between each station

networkStationPairs = stationsWithDifferingGPS.merge(stationsWithDifferingGPS, how='cross', suffixes=('_1', '_2'))

# Remove self-pairings and duplicate pairs
networkStationPairs = networkStationPairs[networkStationPairs['posid_1'] < networkStationPairs['posid_2']]

# Reset the index for clarity
networkStationPairs.reset_index(drop=True, inplace=True)

networkStationPairs["distance"] = networkStationPairs.apply(
    lambda row: distance.distance(
        (row["gps_lat_1"], row["gps_lon_1"]), 
        (row["gps_lat_2"], row["gps_lon_2"])
    ).km, 
    axis=1
)

networkStationPairs = networkStationPairs.drop(columns=["gps_lat_1", "gps_lat_2", "gps_lon_1", "gps_lon_2", "total_amt_1", "total_amt_2"])

In [5]:
networkStationPairs.to_csv('./data/network_station_pairs_without_largest_cities.csv', index=False)

In [4]:
import pandas as pd
import numpy as np
from scipy import stats

from pandas.api.types import CategoricalDtype

stationProximityDataSet: pd.DataFrame = pd.read_csv("data/benz-tr.csv")

stationProximityDataSet = stationProximityDataSet[stationProximityDataSet.country == "cze"]

duplicateSubset = subset=["amt", "day", "hour", "clid"]

stationProximityDataSet = stationProximityDataSet.drop_duplicates(subset=duplicateSubset)

# Drop useless columns
stationProximityDataSet = stationProximityDataSet.drop(columns=["clst", "clid", "day", "hour", "fav", "fav_main", "cl_gps_lat", "cl_gps_lon", "city", "name", "dist", "country"])

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

duplicateSubset = ["gps_lat", "gps_lon"]

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

stationsWithDifferingGPS = stationsWithDifferingGPS.groupby(['gps_lat', 'gps_lon'], as_index=False).agg(
    total_amt=('total_amt', 'sum'),
    posid=('posid', 'first')
)

In [6]:
from geopy import distance

networkStationPairs = stationsWithDifferingGPS.merge(stationsWithDifferingGPS, how='cross', suffixes=('_1', '_2'))

networkStationPairs = networkStationPairs[networkStationPairs['posid_1'] < networkStationPairs['posid_2']]

networkStationPairs.reset_index(drop=True, inplace=True)

networkStationPairs["distance"] = networkStationPairs.apply(
    lambda row: distance.distance(
        (row["gps_lat_1"], row["gps_lon_1"]), 
        (row["gps_lat_2"], row["gps_lon_2"])
    ).km, 
    axis=1
)

networkStationPairs = networkStationPairs.drop(columns=["gps_lat_1", "gps_lat_2", "gps_lon_1", "gps_lon_2", "total_amt_1", "total_amt_2"])

In [7]:
networkStationPairs.to_csv('./data/network_station_pairs_all.csv', index=False)

In [8]:
import pandas as pd
import numpy as np
from scipy import stats

from pandas.api.types import CategoricalDtype

stationProximityDataSet: pd.DataFrame = pd.read_csv("data/benz-tr.csv")

stationProximityDataSet = stationProximityDataSet[stationProximityDataSet.country == "cze"]
stationProximityDataSet = stationProximityDataSet[stationProximityDataSet["city"].str.startswith(("praha"), na=False)]

duplicateSubset = subset=["amt", "day", "hour", "clid"]

stationProximityDataSet = stationProximityDataSet.drop_duplicates(subset=duplicateSubset)

stationProximityDataSet = stationProximityDataSet.drop(columns=["clst", "clid", "day", "hour", "fav", "fav_main", "cl_gps_lat", "cl_gps_lon", "city", "name", "dist", "country"])

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

duplicateSubset = ["gps_lat", "gps_lon"]

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

stationsWithDifferingGPS = stationsWithDifferingGPS.groupby(['gps_lat', 'gps_lon'], as_index=False).agg(
    total_amt=('total_amt', 'sum'),
    posid=('posid', 'first')
)

In [9]:
from geopy import distance

networkStationPairs = stationsWithDifferingGPS.merge(stationsWithDifferingGPS, how='cross', suffixes=('_1', '_2'))

networkStationPairs = networkStationPairs[networkStationPairs['posid_1'] < networkStationPairs['posid_2']]
networkStationPairs.reset_index(drop=True, inplace=True)

networkStationPairs["distance"] = networkStationPairs.apply(
    lambda row: distance.distance(
        (row["gps_lat_1"], row["gps_lon_1"]), 
        (row["gps_lat_2"], row["gps_lon_2"])
    ).km, 
    axis=1
)

networkStationPairs = networkStationPairs.drop(columns=["gps_lat_1", "gps_lat_2", "gps_lon_1", "gps_lon_2", "total_amt_1", "total_amt_2"])

In [10]:
networkStationPairs.to_csv('./data/network_station_pairs_prague_only.csv', index=False)

In [6]:
import pandas as pd
import numpy as np
from scipy import stats

from pandas.api.types import CategoricalDtype

stationProximityDataSet: pd.DataFrame = pd.read_csv("data/benz-tr.csv")

# Drop rows where country is not czechia

stationProximityDataSet = stationProximityDataSet[stationProximityDataSet.country == "cze"]

major_cities = ["praha", "brno", "ostrava", "liberec", "plzen", 
                "olomouc", "budejovice", "hradec kralove"]    

# Cities only

stationProximityDataSet = stationProximityDataSet[stationProximityDataSet['city'].str.lower().isin(major_cities)]

# Lets consider rows which have the same amnt, day, hour, clid to be duplicate and drop them
duplicateSubset = subset=["amt", "day", "hour", "clid"]

stationProximityDataSet = stationProximityDataSet.drop_duplicates(subset=duplicateSubset)

# Drop useless columns
stationProximityDataSet = stationProximityDataSet.drop(columns=["clst", "clid", "day", "hour", "fav", "fav_main", "cl_gps_lat", "cl_gps_lon", "city", "name", "dist", "country"])

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

duplicateSubset = ["gps_lat", "gps_lon"]

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

stationsWithDifferingGPS = stationsWithDifferingGPS.groupby(['gps_lat', 'gps_lon'], as_index=False).agg(
    total_amt=('total_amt', 'sum'),
    posid=('posid', 'first')
)

In [7]:
from geopy import distance

networkStationPairs = stationsWithDifferingGPS.merge(stationsWithDifferingGPS, how='cross', suffixes=('_1', '_2'))

networkStationPairs = networkStationPairs[networkStationPairs['posid_1'] < networkStationPairs['posid_2']]
networkStationPairs.reset_index(drop=True, inplace=True)

networkStationPairs["distance"] = networkStationPairs.apply(
    lambda row: distance.distance(
        (row["gps_lat_1"], row["gps_lon_1"]), 
        (row["gps_lat_2"], row["gps_lon_2"])
    ).km, 
    axis=1
)

networkStationPairs = networkStationPairs.drop(columns=["gps_lat_1", "gps_lat_2", "gps_lon_1", "gps_lon_2", "total_amt_1", "total_amt_2"])

In [8]:
networkStationPairs.to_csv('./data/network_station_pairs_cities_only.csv', index=False)

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

from pandas.api.types import CategoricalDtype

stationProximityDataSet: pd.DataFrame = pd.read_csv("data/benz-tr.csv")

# Drop rows where country is not czechia

stationProximityDataSet = stationProximityDataSet[stationProximityDataSet.country == "cze"]

# Lets consider rows which have the same amnt, day, hour, clid to be duplicate and drop them
duplicateSubset = subset=["amt", "day", "hour", "clid"]

stationProximityDataSet = stationProximityDataSet.drop_duplicates(subset=duplicateSubset)

# Drop useless columns
stationProximityDataSet = stationProximityDataSet.drop(columns=["clst", "clid", "day", "hour", "fav", "fav_main", "cl_gps_lat", "cl_gps_lon", "name", "dist", "country"])

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

duplicateSubset = ["gps_lat", "gps_lon"]

stationsWithDifferingGPS = stationProximityDataSet.groupby("posid", as_index=False).agg(
    total_amt=("amt", "sum"),
    city=("city", "first"),
    gps_lat=("pos_gps_lat", lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    gps_lon=("pos_gps_lon", lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

stationsWithDifferingGPS = stationsWithDifferingGPS.groupby(['gps_lat', 'gps_lon'], as_index=False).agg(
    total_amt=('total_amt', 'sum'),
    posid=('posid', 'first'),
    city=('city', 'first')
)

In [5]:
from geopy.distance import geodesic

major_cities_coords = {
    'praha': (50.073658, 14.418540),
    'brno': (49.195061, 16.606836),
    'ostrava': (49.820923, 18.262524),
    'liberec': (50.766280, 15.054339),
    'plzen': (49.738430, 13.373637),
    'olomouc': (49.593777, 17.250879),
    'budejovice': (48.975658, 14.480255),
    'hradec kralove': (50.210361, 15.825211)
}


radius = 30

def is_near_major_city(lat, lon, rowCity, cities_coords, radius_km):
    for city, coords in cities_coords.items():
        if city == rowCity:
            return True
        if geodesic((lat, lon), coords).km <= radius_km:
            return True
    return False

stationsWithDifferingGPS['near_major_city'] = stationsWithDifferingGPS.apply(
    lambda row: is_near_major_city(row['gps_lat'], row['gps_lon'], row['city'], major_cities_coords, radius),
    axis=1
)

stationsWithDifferingGPS = stationsWithDifferingGPS[stationsWithDifferingGPS['near_major_city']]

stationsWithDifferingGPS = stationsWithDifferingGPS.drop(columns='near_major_city')
stationsWithDifferingGPS

,gps_lat,gps_lon,total_amt,posid,city
10,48.731347,14.631349,53629.00,4413,benesov nad c
12,48.735542,14.483299,306255.60,5767,kaplice
15,48.739560,14.482360,307614.76,5765,kaplice
16,48.742428,14.495017,220383.52,5762,kaplice
46,48.812279,14.322556,336922.25,4851,cesky krumlov
...,...,...,...,...,...
2125,50.913116,15.062570,75521.70,5241,frydlant
2127,50.916020,15.061610,559881.96,5247,frydlant v cechach
2128,50.916240,15.071150,2157.70,5248,frydlant v cechach
2130,50.918812,15.074795,16919.68,5267,frydlant


In [4]:
networkStationPairs.to_csv('./data/network_station_pairs_cities_only.csv', index=False)

NameError: name 'networkStationPairs' is not defined